## Transformer-Related Literature Collection

In [3]:
import os, csv, time, requests, sys
from urllib.parse import urlencode
from tqdm import tqdm

# Calculate the sums
import os, csv, time, requests, xml.etree.ElementTree as ET

import csv
import time
import random
from typing import List

# For scholarly
from scholarly import scholarly

# For unified collectors
from paper_collector.models import Paper
from paper_collector.collectors.scholar import ScholarCollector
from paper_collector.collectors.arxiv import ArxivCollector
from paper_collector.collectors.pubmed import PubMedCollector
from paper_collector.collectors.scopus import ScopusCollector
from paper_collector.utils import deduplicate_papers

In [5]:
QUERY = """(transformer* OR "self-attention" OR BERT OR GPT OR "large language model" OR "retrieval-augmented generation")
AND (prescrib* OR medicat* OR "drug" OR pharmac* OR "medication recommendation" OR prescription)"""

START_DATE = "2020-01-01"
END_DATE = "2025-05-31"

print("Query Loadeds ✔")


Query Loadeds ✔


In [ ]:
def scrape_scholar(query, limit=None, output_file="source/google_scholar_works.csv"):
    print(f"Searching Scholar for: {query}")
    search_query = scholarly.search_pubs(query)

    with open(output_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['title', 'author', 'pub_year', 'venue',
                         'abstract', 'pub_url', 'num_citations', 'cites_id'])
        
        count = 0
        for result in search_query:
            if limit and count >= limit:
                break
            
            bib = result.get('bib', {})
            title = bib.get('title', '')
            author = bib.get('author', [])
            if isinstance(author, list):
                author = '; '.join(author)
                
            row = [
                title,
                author,
                bib.get('pub_year', ''),
                bib.get('venue', ''),
                bib.get('abstract', ''),
                result.get('pub_url', ''),
                result.get('num_citations', 0),
                result.get('cites_id', '')
            ]
            writer.writerow(row)
            count += 1
            
            print(f"✔ Saved {count}: {title[:50]}...")
            time.sleep(random.uniform(1, 3))  # Rate-limit avoidance

    print(f"Finished. {count} records saved → {output_file}")


def collect_from_sources(query, limit=10, output="collected_papers.csv",
                         sources=("scholar", "arxiv", "pubmed", "scopus"),
                         start=START_DATE, end=END_DATE, scopus_key=None):

    collectors = []

    if "scholar" in sources:
        collectors.append(ScholarCollector(start, end))
    if "arxiv" in sources:
        collectors.append(ArxivCollector(start, end))
    if "pubmed" in sources:
        collectors.append(PubMedCollector(start, end))
    if "scopus" in sources:
        if scopus_key:
            collectors.append(ScopusCollector(start, end, api_key=scopus_key))
        else:
            print("⚠ Skipping Scopus (missing API key)")

    collected = []
    for c in collectors:
        try:
            print(f"\nCollecting from {c.__class__.__name__}...")
            collected.extend(c.collect(query, limit=limit))
        except Exception as e:
            print(f"❌ Error in {c.__class__.__name__}: {e}")

    # Deduplicate by DOI or title
    unique = deduplicate_papers(collected)

    # Save
    if unique:
        keys = unique[0].to_dict().keys()
        with open(output, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=keys)
            writer.writeheader()
            for p in unique:
                writer.writerow(p.to_dict())

        print(f"\n📦 Saved {len(unique)} unique papers → {output}")
    else:
        print("No papers collected.")



In [ ]:
# scrape_scholar(QUERY, limit=20)  # adjust limit as needed

collect_from_sources(
    QUERY,
    limit=20,
    output="papers_all_sources.csv",
    sources=["scholar", "arxiv", "pubmed"],  # safe default
)



🟦 Section 1 — OpenAlex Metadata Collection

Retrieves results using the OpenAlex Works API, including:
DOI\n
Publication year/date
Venue metadata
Open-access status
Reconstructed abstract (if available)
Authors

In [ ]:
MAILTO = "bukhari.453@s.kyushu-u.ac.jp"
OUT_CSV = "openalex_works.csv"

QUERY_HUMAN = r'''(transformer* OR "self-attention" OR BERT OR GPT OR "large language model" OR "retrieval-augmented generation") AND (prescrib* OR medicat* OR "drug" OR pharmac* OR "medication recommendation" OR prescription)'''.strip()
DATE_FROM, DATE_TO = "2020-01-01", "2025-05-31"

BASE = "https://api.openalex.org/works"

# We search fulltext fields, then filter by date, language, and has_doi if you like
params = {
    "search": QUERY_HUMAN,
    "filter": f"from_publication_date:{DATE_FROM},to_publication_date:{DATE_TO}",
    "per_page": 200,
    "cursor": "*",
    "mailto": MAILTO,
}

fields = ["id", "doi", "title", "publication_year", "publication_date",
          "authorships", "host_venue", "type", "open_access", "cited_by_count", "abstract_inverted_index",
          "primary_topic"]

with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["source", "work_id", "doi", "title", "year", "date", "venue", "type", "is_oa", "oa_status", "cited_by",
                "abstract", "authors"])
    total = 0
    while True:
        r = requests.get(BASE, params=params, timeout=60)
        r.raise_for_status()
        j = r.json()
        results = j.get("results", [])
        if not results: break
        for x in results:
            doi = (x.get("doi") or "").lower().replace("https://doi.org/", "")
            title = x.get("title") or ""
            year = x.get("publication_year") or ""
            date = x.get("publication_date") or ""
            venue = (x.get("host_venue") or {}).get("display_name") or ""
            wtype = x.get("type") or ""
            oa = x.get("open_access") or {}
            is_oa = oa.get("is_oa")
            oa_status = oa.get("oa_status") or ""
            cited_by = x.get("cited_by_count") or 0
            # Recompose abstract text if present
            inv = x.get("abstract_inverted_index")
            abstract = ""
            if isinstance(inv, dict):
                words = sorted([(pos, word) for word, poss in inv.items() for pos in poss])
                abstract = " ".join(w for _, w in words)
            # Simple author string
            authors = "; ".join(
                [(a.get("author", {}) or {}).get("display_name", "") for a in x.get("authorships") or []])
            w.writerow(
                ["OpenAlex", x.get("id"), doi, title, year, date, venue, wtype, is_oa, oa_status, cited_by, abstract,
                 authors])
            total += 1
        params["cursor"] = j.get("meta", {}).get("next_cursor")
        if not params["cursor"]: break
        time.sleep(0.1)

print(f"Saved {total} OpenAlex records to {OUT_CSV}")

Saved 19095 OpenAlex records to openalex_works.csv


In [ ]:


EMAIL = "your.email@institution.edu"  # NCBI asks for email/tool id
TOOL = "your_tool_name"

QUERY_HUMAN = r'''(transformer*[Title/Abstract] OR "self-attention"[Title/Abstract] OR BERT[Title/Abstract] OR GPT[Title/Abstract] OR "large language model"[Title/Abstract] OR "retrieval-augmented generation"[Title/Abstract]) AND (prescrib*[Title/Abstract] OR medicat*[Title/Abstract] OR drug[Title/Abstract] OR pharmac*[Title/Abstract] OR "medication recommendation"[Title/Abstract] OR prescription[Title/Abstract])'''.strip()
DATE_FROM, DATE_TO = "2020/01/01", "2025/05/31"  # PubMed date format

OUT_CSV = "pubmed_works.csv"


def esearch(term, retmax=100000):
    base = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    params = {
        "db": "pubmed", "term": term,
        "datetype": "pdat", "mindate": DATE_FROM, "maxdate": DATE_TO,
        "retmax": retmax, "retmode": "json",
        "usehistory": "y", "tool": TOOL, "email": EMAIL
    }
    r = requests.get(base, params=params, timeout=60)
    r.raise_for_status()
    return r.json()


def efetch(webenv, query_key, retstart, retmax=200):
    base = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    params = {
        "db": "pubmed", "query_key": query_key, "WebEnv": webenv,
        "retstart": retstart, "retmax": retmax,
        "retmode": "xml", "tool": TOOL, "email": EMAIL
    }
    r = requests.get(base, params=params, timeout=60)
    r.raise_for_status()
    return r.text


def get_text(elem, path):
    x = elem.find(path)
    return x.text if x is not None and x.text else ""


def parse_article(article):
    med = article.find("MedlineCitation")
    pmid = get_text(med, "PMID")
    art = med.find("Article")
    title = get_text(art, "ArticleTitle")
    # Abstract (concat sections)
    abstract = " ".join([t.text for t in art.findall("Abstract/AbstractText") if t is not None and t.text]) or ""
    # Year
    y = get_text(art, "Journal/JournalIssue/PubDate/Year")
    if not y:
        y = get_text(art, "Journal/JournalIssue/PubDate/MedlineDate")[:4]
    # Journal
    journal = get_text(art, "Journal/Title")
    # DOI
    doi = ""
    for aid in art.findall("ELocationID"):
        if aid.get("EIdType", "").lower() == "doi" and aid.text:
            doi = aid.text.lower().strip()
            break
    if not doi:
        for el in med.findall("ArticleIdList/ArticleId"):
            if el.get("IdType", "").lower() == "doi" and el.text:
                doi = el.text.lower().strip()
                break
    # MeSH
    mesh_terms = [get_text(x, "DescriptorName") for x in med.findall("MeshHeadingList/MeshHeading")]
    authors = []
    for a in art.findall("AuthorList/Author"):
        last = get_text(a, "LastName");
        fore = get_text(a, "ForeName")
        if last or fore: authors.append((" ".join([fore, last])).strip())
    return {
        "source": "PubMed", "pmid": pmid, "doi": doi, "title": title, "year": y, "date": "", "venue": journal,
        "type": "", "is_oa": "", "oa_status": "", "cited_by": "",
        "abstract": abstract, "authors": "; ".join(authors), "mesh": "; ".join(mesh_terms)
    }


# Run
j = esearch(QUERY_HUMAN)
webenv = j["esearchresult"]["webenv"]
qk = j["esearchresult"]["querykey"]
count = int(j["esearchresult"]["count"])
print("PubMed hits:", count)

with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    import csv

    w = csv.DictWriter(f, fieldnames=["source", "pmid", "doi", "title", "year", "date", "venue", "type", "is_oa",
                                      "oa_status", "cited_by", "abstract", "authors", "mesh"])
    w.writeheader()
    for start in tqdm(range(0, count, 200)):
        xml = efetch(webenv, qk, start, 200)
        root = ET.fromstring(xml)
        for art in root.findall("PubmedArticle"):
            rec = parse_article(art)
            w.writerow(rec)
        time.sleep(0.34)  # be polite to NCBI
print(f"Saved {count} PubMed records to {OUT_CSV}")


## Data Collection for arxiv.

In [ ]:
import arxiv
import pandas as pd
import requests
import datetime
import time

# --- CONFIGURATION ---
QUERY = "LLM"  # Your search keyword
MAX_RESULTS = 10  # Number of papers to fetch
# Optional: Add your Semantic Scholar API Key here if you have one to avoid rate limits.
# Get free key: https://www.semanticscholar.org/product/api
S2_API_KEY = None

def get_semantic_scholar_data(arxiv_id, api_key=None):
    """
    Fetches citation count and venue info from Semantic Scholar API.
    """
    url = f"https://api.semanticscholar.org/graph/v1/paper/ARXIV:{arxiv_id}"
    params = {'fields': 'citationCount,venue,url,year'}
    headers = {}
    if api_key:
        headers = {'x-api-key': api_key}

    try:
        # 1 second sleep to be polite to the public API (rate limit is tight without key)
        if not api_key:
            time.sleep(1.0)

        response = requests.get(url, params=params, headers=headers, timeout=5)
        if response.status_code == 200:
            return response.json()
        else:
            return None
    except Exception as e:
        print(f"Error fetching S2 data for {arxiv_id}: {e}")
        return None

def fetch_papers(query, max_results=10):
    # Construct the arXiv client
    client = arxiv.Client()

    search = arxiv.Search(
        query=query,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.Relevance
    )

    papers_data = []

    print(f"Fetching {max_results} papers for query: '{query}'...")

    for result in client.results(search):
        # --- 1. Basic Metadata from arXiv ---
        arxiv_id = result.get_short_id().split('v')[0] # Remove version (e.g., 2301.0001v1 -> 2301.0001)
        title = result.title
        authors_list = [author.name for author in result.authors]
        authors_str = ", ".join(authors_list)
        pub_date = result.published
        year = pub_date.year

        # Calculate Age (Years since publication)
        current_year = datetime.datetime.now().year
        age = current_year - year
        age = age if age > 0 else 0.5  # Avoid division by zero

        # --- 2. Advanced Data from Semantic Scholar (Citations) ---
        s2_data = get_semantic_scholar_data(arxiv_id, S2_API_KEY)

        citations = 0
        s2_url = "N/A"
        publisher = "arXiv" # Default

        if s2_data:
            citations = s2_data.get('citationCount', 0)
            s2_url = s2_data.get('url', "N/A")
            venue = s2_data.get('venue', "")
            if venue:
                publisher = venue

        # Calculate Cites Per Year (Average)
        cites_per_year = round(citations / age, 2)
        cites_per_author = round(citations / len(authors_list), 2) if authors_list else 0

        # --- 3. Map to Requested Columns ---
        paper_info = {
            "Cites": citations,
            "Authors": authors_str,
            "Title": title,
            "Year": year,
            "Source": "arXiv",
            "Publisher": publisher,  # arXiv or Venue from S2
            "ArticleURL": result.entry_id,
            "CitesURL": s2_url,      # Link to Semantic Scholar page
            "GSRank": "N/A",         # Cannot scrape Google Scholar (ToS violation)
            "QueryDate": datetime.datetime.now().strftime("%Y-%m-%d"),
            "Type": "Preprint",
            "DOI": result.doi if result.doi else "N/A",
            "ISSN": "N/A",           # arXiv does not have an ISSN
            "CitationURL": s2_url,
            "Volume": "N/A",         # Only available if published in journal
            "Issue": "N/A",
            "StartPage": "N/A",
            "EndPage": "N/A",
            "ECC": "N/A",            # Expected Citation Count is proprietary (Scopus)
            "CitesPerYear": cites_per_year,
            "CitesPerAuthor": cites_per_author,
            "AuthorCount": len(authors_list),
            "Age": round(age, 1),
            "Abstract": result.summary.replace("\n", " "),
            "FullTextURL": result.pdf_url,
            "RelatedURL": s2_url
        }

        papers_data.append(paper_info)
        print(f"Processed: {title[:50]}...")

    return pd.DataFrame(papers_data)

# --- EXECUTION ---
if __name__ == "__main__":
    df = fetch_papers(QUERY, MAX_RESULTS)

    # Save to CSV
    filename = "arxiv_papers_complete.csv"
    df.to_csv(filename, index=False)

    print(f"\nSuccessfully saved {len(df)} papers to {filename}")
    print(df[['Title', 'Cites', 'Year']].head())